# CD Project: Compiler-Style Lexical Analysis of Comic Dialogue Text
**Course:** Compiler Design Lab (DA 6)  
**Team No:** 11 | **Slot:** G2+TG2  
**Team Members:** Kartikeya Behl (24BAI0083), Samarth Pritmani (24BAI0078), Vayun Tejwani (24BAI0027)

### Architecture Pipeline (Front-End Stages):
`Dialogue Input` ➔ **Module 1 (Speaker Split)** ➔ **Module 2 (Cleaning)** ➔ **Module 3 (Lexer)** ⟷ **Module 4 (Symbol Table)**

> This notebook implements module-by-module execution up to the Symbol Table (Lexer) as required, and supports processing input **one line at a time**.


---
## Module 1: Speaker Split
* **Function**: Takes raw dialogue input (from transcribed speech bubbles or script files) and extracts the speaker prefix (`SPEAKER: line` or `[SPEAKER]: line`) from the dialogue utterance.
* **Input**: Raw comic dialogue string with or without speaker prefix.
* **Output**: Identified Speaker and separated dialogue body.


In [1]:
import re

class SpeakerSplitter:
    def __init__(self):
        # Regex matches 'SPEAKER: line' or '[SPEAKER]: line'
        self.pattern = re.compile(
            r'^\s*(?:\[(?P<bracket_spk>[^\]]+)\]|(?P<speaker>[A-Za-z0-9_\-\.\s]+?))\s*:\s*(?P<dialogue>.*)$'
        )

    def split(self, raw_line: str):
        match = self.pattern.match(raw_line)
        if match:
            speaker = match.group("bracket_spk") or match.group("speaker")
            dialogue = match.group("dialogue")
            return speaker.strip(), dialogue.strip()
        return "UNKNOWN_SPEAKER", raw_line.strip()

splitter = SpeakerSplitter()

# ----------------- MODULE 1 INPUT / OUTPUT DEMONSTRATION -----------------
test_inputs_m1 = [
    "CLINT: WHERE COULD HE BE, NATASHA?",
    "BATMAN: HAVE A PEANUT.",
    "GREEN ARROW: I JUST STOPPED THE ENTIRE WORLD FROM BEING EATEN, DINAH.",
    "GORDON: BATMAN~ WE HAVE A 10-99 AT SECTOR 4#."
]

print("=" * 75)
print("              MODULE 1: SPEAKER SPLIT - INPUT & OUTPUT")
print("=" * 75)
for line in test_inputs_m1:
    spk, text = splitter.split(line)
    print(f"INPUT (Raw Line)     : {line}")
    print(f"OUTPUT (Speaker Tag) : '{spk}'")
    print(f"OUTPUT (Dialogue)    : '{text}'")
    print("-" * 75)


              MODULE 1: SPEAKER SPLIT - INPUT & OUTPUT
INPUT (Raw Line)     : CLINT: WHERE COULD HE BE, NATASHA?
OUTPUT (Speaker Tag) : 'CLINT'
OUTPUT (Dialogue)    : 'WHERE COULD HE BE, NATASHA?'
---------------------------------------------------------------------------
INPUT (Raw Line)     : BATMAN: HAVE A PEANUT.
OUTPUT (Speaker Tag) : 'BATMAN'
OUTPUT (Dialogue)    : 'HAVE A PEANUT.'
---------------------------------------------------------------------------
INPUT (Raw Line)     : GREEN ARROW: I JUST STOPPED THE ENTIRE WORLD FROM BEING EATEN, DINAH.
OUTPUT (Speaker Tag) : 'GREEN ARROW'
OUTPUT (Dialogue)    : 'I JUST STOPPED THE ENTIRE WORLD FROM BEING EATEN, DINAH.'
---------------------------------------------------------------------------
INPUT (Raw Line)     : GORDON: BATMAN~ WE HAVE A 10-99 AT SECTOR 4#.
OUTPUT (Speaker Tag) : 'GORDON'
OUTPUT (Dialogue)    : 'BATMAN~ WE HAVE A 10-99 AT SECTOR 4#.'
---------------------------------------------------------------------------


---
## Module 2: Cleaning & Normalization
* **Function**: Normalizes character spacing, collapses redundant spaces/tabs, and repairs hyphenated words split across line breaks (e.g., `SEC- TOR` $\rightarrow$ `SECTOR`).
* **Input**: Extracted dialogue text from Module 1.
* **Output**: Standardized, continuous dialogue string ready for tokenization.


In [2]:
class DialogueCleaner:
    def clean(self, text: str) -> str:
        # 1. Repair hyphenated broken words across line-breaks (e.g. 'SEC- TOR' -> 'SECTOR')
        text = re.sub(r'(\b[A-Za-z]+)-\s+([A-Za-z]+\b)', r'\1\2', text)
        # 2. Collapse multiple spaces and tabs
        text = re.sub(r'[ \t]+', ' ', text)
        return text.strip()

cleaner = DialogueCleaner()

# ----------------- MODULE 2 INPUT / OUTPUT DEMONSTRATION -----------------
test_inputs_m2 = [
    "WHERE   COULD HE   BE,   NATASHA?",
    "HAVE   A    PEANUT.",
    "BATMAN~   WE HAVE A 10-99   AT   SEC- TOR 4#."
]

print("=" * 75)
print("              MODULE 2: CLEANING - INPUT & OUTPUT")
print("=" * 75)
for line in test_inputs_m2:
    cleaned = cleaner.clean(line)
    print(f"INPUT (Uncleaned) : {line}")
    print(f"OUTPUT (Cleaned)  : {cleaned}")
    print("-" * 75)


              MODULE 2: CLEANING - INPUT & OUTPUT
INPUT (Uncleaned) : WHERE   COULD HE   BE,   NATASHA?
OUTPUT (Cleaned)  : WHERE COULD HE BE, NATASHA?
---------------------------------------------------------------------------
INPUT (Uncleaned) : HAVE   A    PEANUT.
OUTPUT (Cleaned)  : HAVE A PEANUT.
---------------------------------------------------------------------------
INPUT (Uncleaned) : BATMAN~   WE HAVE A 10-99   AT   SEC- TOR 4#.
OUTPUT (Cleaned)  : BATMAN~ WE HAVE A 10-99 AT SECTOR 4#.
---------------------------------------------------------------------------


---
## Module 3: Comic Lexer (Lexical Analysis & OCR Error Tracking)
* **Function**: Scans cleaned dialogue characters from left-to-right using regular expressions. Categorizes characters into 9 token classes: `WORD`, `NUMBER`, `CONTRACTION`, `PUNCT`, `TERMINATOR`, `ELLIPSIS` (`...`), `INTERROBANG` (`?!`), `SOUND_FX` (`BOOM`, `AAARGH`), and `UNKNOWN`.
* **OCR Error Interception**: Any character matching no grammatical rule (e.g., stray OCR artifacts `~`, `#`) is trapped as an **UNKNOWN** token and recorded as a **Lexical Error** with its exact character position offset.
* **Input**: Cleaned dialogue text from Module 2.
* **Output**: Token Stream (Type, Value, Character Span) + Lexical Error Audit.


In [3]:
from enum import Enum

class TokenType(str, Enum):
    WORD = "WORD"
    NUMBER = "NUMBER"
    CONTRACTION = "CONTRACTION"
    PUNCT = "PUNCT"
    TERMINATOR = "TERMINATOR"
    ELLIPSIS = "ELLIPSIS"
    INTERROBANG = "INTERROBANG"
    SOUND_FX = "SOUND_FX"
    UNKNOWN = "UNKNOWN"

# Onomatopoeia & sound effects lexicon
SOUND_EFFECTS = {
    "BOOM", "AAARGH", "AARGH", "POW", "BAM", "ZAP", "SMASH",
    "THWIP", "OOF", "UGH", "CRASH", "BANG", "KABOOM"
}

class ComicLexer:
    def __init__(self):
        self.re_ellipsis = re.compile(r'^(?:\.{3,}|…)')
        self.re_interrobang = re.compile(r'^(?:\?+!+|!+\?+)[\?!]*')
        self.re_contraction = re.compile(r"^[A-Za-z]+['’][A-Za-z]+")
        self.re_word = re.compile(r'^[A-Za-z]+')
        self.re_number = re.compile(r'^\d+')
        self.re_terminator = re.compile(r'^[.?!]')
        self.re_punct = re.compile(r'^[,:;\-—]')
        # Detects elongated shouting vowels (e.g. AAARGH, NOOOO, YEAAAH)
        self.re_elongated = re.compile(r'([A-Za-z])\1{2,}', re.IGNORECASE)

    def tokenize(self, text: str):
        tokens = []
        errors = []
        i = 0
        n = len(text)

        while i < n:
            ch = text[i]
            # Skip whitespace
            if ch.isspace():
                i += 1
                continue

            sub = text[i:]

            # 1. Ellipsis (... or …)
            m = self.re_ellipsis.match(sub)
            if m:
                v = m.group(0)
                tokens.append((TokenType.ELLIPSIS, v, i, i + len(v)))
                i += len(v)
                continue

            # 2. Interrobang (?!, !?)
            m = self.re_interrobang.match(sub)
            if m:
                v = m.group(0)
                tokens.append((TokenType.INTERROBANG, v, i, i + len(v)))
                i += len(v)
                continue

            # 3. Contraction (DON'T, IT'S)
            m = self.re_contraction.match(sub)
            if m:
                v = m.group(0)
                tokens.append((TokenType.CONTRACTION, v, i, i + len(v)))
                i += len(v)
                continue

            # 4. Words & Sound FX
            m = self.re_word.match(sub)
            if m:
                v = m.group(0)
                t_type = TokenType.SOUND_FX if (v.upper() in SOUND_EFFECTS or self.re_elongated.search(v)) else TokenType.WORD
                tokens.append((t_type, v, i, i + len(v)))
                i += len(v)
                continue

            # 5. Numeric Literals
            m = self.re_number.match(sub)
            if m:
                v = m.group(0)
                tokens.append((TokenType.NUMBER, v, i, i + len(v)))
                i += len(v)
                continue

            # 6. Punctuation
            m = self.re_punct.match(sub)
            if m:
                v = m.group(0)
                tokens.append((TokenType.PUNCT, v, i, i + len(v)))
                i += len(v)
                continue

            # 7. Terminator
            m = self.re_terminator.match(sub)
            if m:
                v = m.group(0)
                tokens.append((TokenType.TERMINATOR, v, i, i + len(v)))
                i += len(v)
                continue

            # 8. Unrecognized Symbol -> Lexical Error (OCR Noise)
            errors.append({"symbol": ch, "position": i})
            tokens.append((TokenType.UNKNOWN, ch, i, i + 1))
            i += 1

        return tokens, errors

lexer = ComicLexer()

# ----------------- MODULE 3 INPUT / OUTPUT DEMONSTRATION -----------------
test_sample_m3 = "BATMAN~ WE HAVE A 10-99 AT SECTOR 4#. BOOM! DON'T WAIT...?!"
tokens_m3, errors_m3 = lexer.tokenize(test_sample_m3)

print("=" * 80)
print("              MODULE 3: COMIC LEXER - INPUT & OUTPUT")
print("=" * 80)
print(f"INPUT (Cleaned Text) : {test_sample_m3}")
print("-" * 80)
print("OUTPUT (Token Stream):")
print(f"{'INDEX':<7}{'CLASS':<15}{'TOKEN VALUE':<18}{'SPAN':<10}")
print("-" * 55)
for idx, (t_type, val, s, e) in enumerate(tokens_m3, 1):
    print(f"{idx:<7}{t_type.value:<15}'{val}'{s:>10}:{e:<4}")

print("-" * 80)
print(f"OUTPUT (Lexical Errors / Stray OCR Noise) [{len(errors_m3)} Detected]:")
for err in errors_m3:
    print(f"  • Stray Symbol '{err['symbol']}' intercepted at character coordinate {err['position']}")
print("=" * 80)


              MODULE 3: COMIC LEXER - INPUT & OUTPUT
INPUT (Cleaned Text) : BATMAN~ WE HAVE A 10-99 AT SECTOR 4#. BOOM! DON'T WAIT...?!
--------------------------------------------------------------------------------
OUTPUT (Token Stream):
INDEX  CLASS          TOKEN VALUE       SPAN      
-------------------------------------------------------
1      WORD           'BATMAN'         0:6   
2      UNKNOWN        '~'         6:7   
3      WORD           'WE'         8:10  
4      WORD           'HAVE'        11:15  
5      WORD           'A'        16:17  
6      NUMBER         '10'        18:20  
7      PUNCT          '-'        20:21  
8      NUMBER         '99'        21:23  
9      WORD           'AT'        24:26  
10     WORD           'SECTOR'        27:33  
11     NUMBER         '4'        34:35  
12     UNKNOWN        '#'        35:36  
13     TERMINATOR     '.'        36:37  
14     SOUND_FX       'BOOM'        38:42  
15     TERMINATOR     '!'        42:43  
16     CONTRACTION

---
## Module 4: Symbol Table
* **Function**: Maintains the central repository of distinct tokens. Tracks category, occurrence frequency count, and flags candidate proper names for downstream character popularity analysis.
* **Input**: Token stream emitted by Module 3 (Lexer).
* **Output**: Formatted Symbol Table with token classes, cumulative counts, and proper name flags.


In [4]:
class SymbolTable:
    def __init__(self):
        # Key: uppercase token identifier
        self.table = {}

    def insert(self, token_val: str, token_type: str, is_proper_name: bool = False):
        key = token_val.upper()
        if key not in self.table:
            self.table[key] = {
                "name": token_val,
                "type": token_type,
                "count": 1,
                "is_proper_name": is_proper_name
            }
        else:
            self.table[key]["count"] += 1
            if is_proper_name:
                self.table[key]["is_proper_name"] = True

    def display(self):
        print(f"{'TOKEN':<16}{'LEXICAL CLASS':<16}{'FREQUENCY':<12}{'PROPER NAME?':<15}")
        print("-" * 60)
        # Display sorted by frequency descending
        sorted_entries = sorted(self.table.values(), key=lambda x: x['count'], reverse=True)
        for entry in sorted_entries:
            print(f"{entry['name']:<16}{entry['type']:<16}{entry['count']:<12}{str(entry['is_proper_name']):<15}")

# ----------------- MODULE 4 INPUT / OUTPUT DEMONSTRATION -----------------
KNOWN_NAMES = {"BATMAN", "NATASHA", "BRUCE", "DINAH", "CLINT", "JOKER", "SUPERMAN"}

symbol_table = SymbolTable()

# Insert tokens from sample sentences to demonstrate accumulation
sample_stream = [
    "CLINT: WHERE COULD HE BE, NATASHA?",
    "BATMAN: HAVE A PEANUT.",
    "BATMAN: THE JOKER IS AT THE DOCKS, ROBIN."
]

for line in sample_stream:
    _, text = splitter.split(line)
    cleaned = cleaner.clean(text)
    toks, _ = lexer.tokenize(cleaned)
    for t_type, val, _, _ in toks:
        is_name = val.upper() in KNOWN_NAMES
        symbol_table.insert(token_val=val, token_type=t_type.value, is_proper_name=is_name)

print("=" * 75)
print("            MODULE 4: SYMBOL TABLE - CUMULATIVE SNAPSHOT")
print("=" * 75)
symbol_table.display()
print("=" * 75)


            MODULE 4: SYMBOL TABLE - CUMULATIVE SNAPSHOT
TOKEN           LEXICAL CLASS   FREQUENCY   PROPER NAME?   
------------------------------------------------------------
,               PUNCT           2           False          
.               TERMINATOR      2           False          
THE             WORD            2           False          
WHERE           WORD            1           False          
COULD           WORD            1           False          
HE              WORD            1           False          
BE              WORD            1           False          
NATASHA         WORD            1           True           
?               TERMINATOR      1           False          
HAVE            WORD            1           False          
A               WORD            1           False          
PEANUT          WORD            1           False          
JOKER           WORD            1           True           
IS              WORD            1         

---
## Live Interactive Pipeline: Process Input One Line at a Time
Run this cell to enter any dialogue line (or press Enter to execute the default sample).  
It coordinates **Module 1 ➔ Module 2 ➔ Module 3 ⟷ Module 4** and displays the step-by-step inputs and outputs.


In [5]:
def process_single_line_step_by_step(raw_line: str, sym_tab: SymbolTable):
    print("=" * 80)
    print(f"RAW INPUT LINE : \"{raw_line}\"")
    print("=" * 80)

    # STEP 1: Speaker Split
    spk, raw_dialogue = splitter.split(raw_line)
    print("\n[MODULE 1 OUTPUT: SPEAKER SPLIT]")
    print(f"  • Speaker Tag Extracted : '{spk}'")
    print(f"  • Dialogue Utterance    : \"{raw_dialogue}\"")

    # STEP 2: Cleaning
    cleaned_dialogue = cleaner.clean(raw_dialogue)
    print("\n[MODULE 2 OUTPUT: CLEANING & NORMALIZATION]")
    print(f"  • Standardized Text     : \"{cleaned_dialogue}\"")

    # STEP 3: Lexer
    tokens, errors = lexer.tokenize(cleaned_dialogue)
    short_classes = " ".join(t[0].value for t in tokens)
    print("\n[MODULE 3 OUTPUT: COMIC LEXER]")
    print(f"  • Short Form Token Pattern : {short_classes}")
    print(f"  • Total Tokens Emitted     : {len(tokens)}")
    print("  • Detailed Tokens List     :")
    for idx, (t_type, val, s, e) in enumerate(tokens, 1):
        print(f"      {idx:>2}. [{t_type.value:<12}] '{val}' (span {s}:{e})")

    if errors:
        print(f"  • Lexical Errors Intercepted : {len(errors)} OCR stray symbol(s)")
        for err in errors:
            print(f"      (!) Symbol '{err['symbol']}' at character offset {err['position']}")
    else:
        print("  • Lexical Errors Intercepted : 0 (Clean lexical scan)")

    # STEP 4: Symbol Table Update
    for t_type, val, _, _ in tokens:
        is_name = val.upper() in KNOWN_NAMES
        sym_tab.insert(val, t_type.value, is_proper_name=is_name)

    print("\n[MODULE 4 OUTPUT: UPDATED SYMBOL TABLE SNAPSHOT]")
    sym_tab.display()
    print("=" * 80 + "\n")

# --- Interactive Line Input ---
# In Google Colab, you can type any sentence in the input box:
interactive_sym_tab = SymbolTable()
default_sample = "CLINT: WHERE COULD HE BE, NATASHA?"

# Use default for pre-rendered view, or uncomment input() for manual typing in Colab:
# user_line = input(f"Enter dialogue line [Press Enter for '{default_sample}']: ").strip() or default_sample
user_line = default_sample

process_single_line_step_by_step(user_line, interactive_sym_tab)


RAW INPUT LINE : "CLINT: WHERE COULD HE BE, NATASHA?"

[MODULE 1 OUTPUT: SPEAKER SPLIT]
  • Speaker Tag Extracted : 'CLINT'
  • Dialogue Utterance    : "WHERE COULD HE BE, NATASHA?"

[MODULE 2 OUTPUT: CLEANING & NORMALIZATION]
  • Standardized Text     : "WHERE COULD HE BE, NATASHA?"

[MODULE 3 OUTPUT: COMIC LEXER]
  • Short Form Token Pattern : WORD WORD WORD WORD PUNCT WORD TERMINATOR
  • Total Tokens Emitted     : 7
  • Detailed Tokens List     :
       1. [WORD        ] 'WHERE' (span 0:5)
       2. [WORD        ] 'COULD' (span 6:11)
       3. [WORD        ] 'HE' (span 12:14)
       4. [WORD        ] 'BE' (span 15:17)
       5. [PUNCT       ] ',' (span 17:18)
       6. [WORD        ] 'NATASHA' (span 19:26)
       7. [TERMINATOR  ] '?' (span 26:27)
  • Lexical Errors Intercepted : 0 (Clean lexical scan)

[MODULE 4 OUTPUT: UPDATED SYMBOL TABLE SNAPSHOT]
TOKEN           LEXICAL CLASS   FREQUENCY   PROPER NAME?   
------------------------------------------------------------
WHERE       

---
## Benchmark Demonstration: Processing Table 2 Inputs (One-by-One)
This cell processes dialogue sentences from Table 2 of the base paper along with OCR noise samples, demonstrating sequential module outputs.


In [6]:
benchmark_dialogues = [
    "BATMAN: HAVE A PEANUT.",
    "GREEN ARROW: I JUST STOPPED THE ENTIRE WORLD FROM BEING EATEN, DINAH.",
    "GORDON: BATMAN~ WE HAVE A 10-99 AT SECTOR 4#.",
    "THE JOKER: HAHAHA! DID YOU REALLY THINK YOU COULD CATCH ME, BRUCE?!"
]

bench_sym_tab = SymbolTable()
for dialogue in benchmark_dialogues:
    process_single_line_step_by_step(dialogue, bench_sym_tab)


RAW INPUT LINE : "BATMAN: HAVE A PEANUT."

[MODULE 1 OUTPUT: SPEAKER SPLIT]
  • Speaker Tag Extracted : 'BATMAN'
  • Dialogue Utterance    : "HAVE A PEANUT."

[MODULE 2 OUTPUT: CLEANING & NORMALIZATION]
  • Standardized Text     : "HAVE A PEANUT."

[MODULE 3 OUTPUT: COMIC LEXER]
  • Short Form Token Pattern : WORD WORD WORD TERMINATOR
  • Total Tokens Emitted     : 4
  • Detailed Tokens List     :
       1. [WORD        ] 'HAVE' (span 0:4)
       2. [WORD        ] 'A' (span 5:6)
       3. [WORD        ] 'PEANUT' (span 7:13)
       4. [TERMINATOR  ] '.' (span 13:14)
  • Lexical Errors Intercepted : 0 (Clean lexical scan)

[MODULE 4 OUTPUT: UPDATED SYMBOL TABLE SNAPSHOT]
TOKEN           LEXICAL CLASS   FREQUENCY   PROPER NAME?   
------------------------------------------------------------
HAVE            WORD            1           False          
A               WORD            1           False          
PEANUT          WORD            1           False          
.               TERMI